In [8]:
import numpy as np
import random
import time
from IPython.display import clear_output

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

In [9]:
GRID = [
    ['S', 'F', 'F'],   # y=0
    ['F', 'H', 'F'],   # y=1
    ['F', 'F', 'G'],   # y=2
]

N_STATES  = 9
N_ACTIONS = 4
HOLES     = {4}        # (x=1, y=1)
GOAL      = 8          # (x=2, y=2)
START     = 0          # (x=0, y=0)

ACTION_SYMBOLS = {0: '←', 1: '↓', 2: '→', 3: '↑'}
ACTION_NAMES   = {0: 'Left', 1: 'Down', 2: 'Right', 3: 'Up'}

def xy_to_state(x, y):
    """(x=col, y=row) → flat state index."""
    return y * 3 + x

def state_to_xy(s):
    """Flat state index → (x=col, y=row)."""
    return s % 3, s // 3

def step(state, action):
    """Deterministic step. Returns (next_state, reward, done)."""
    x, y = state_to_xy(state)
    if   action == 0: x = max(x - 1, 0)   # Left
    elif action == 1: y = min(y + 1, 2)   # Down
    elif action == 2: x = min(x + 1, 2)   # Right
    elif action == 3: y = max(y - 1, 0)   # Up
    ns = xy_to_state(x, y)
    if ns in HOLES: return ns, -1.0, True
    if ns == GOAL:  return ns,  1.0, True
    return ns, 0.0, False

In [10]:
def print_optimal_q_matrix(q_table, episode=None, changed_state=None):
    """
    Print the Q-matrix with:
      - x-coordinates (0,1,2) across the top
      - y-coordinates (0,1,2) down the left side
      - Each cell = best Q value across all 4 actions + best action arrow
      - The cell that was just updated is marked with [  ]
    """
    title = f"OPTIMAL Q-MATRIX  (episode {episode})" if episode is not None \
            else "OPTIMAL Q-MATRIX"
    print(f"\n{'═'*52}")
    print(f"  {title}")
    print(f"  Each cell: best Q value  +  best action")
    print(f"{'═'*52}")

    CELL_W = 14

    # Header row — x labels
    print(f"{'':>8}", end='')
    for x in range(3):
        print(f"{'x = ' + str(x):^{CELL_W}}", end='')
    print()
    print(f"{'':>8}" + '─' * (CELL_W * 3))

    for y in range(3):
        # Row label
        print(f"{'y = '+str(y):>7} |", end='')
        for x in range(3):
            s = xy_to_state(x, y)
            cell_type = GRID[y][x]

            if s in HOLES:
                content = '  HOLE  '
            elif s == GOAL:
                best_q = np.max(q_table[s])
                content = f'  G {best_q:+.3f}'
            else:
                best_a  = np.argmax(q_table[s])
                best_q  = q_table[s][best_a]
                arrow   = ACTION_SYMBOLS[best_a]
                content = f'{arrow} {best_q:+.4f}'

            # Highlight the updated cell
            if s == changed_state:
                cell_str = f'[{content:^{CELL_W-2}}]'
            else:
                cell_str = f' {content:^{CELL_W-2}} '

            print(cell_str, end='')
        print()

    print('─' * (CELL_W * 3 + 8))


def print_optimal_q_matrix_pandas(q_table, episode=None, changed_state=None):
    """
    Pandas-styled version: colour-coded heatmap in Jupyter/Colab output.
    Falls back to plain text if pandas is not available.
    """
    if not HAS_PANDAS:
        print_optimal_q_matrix(q_table, episode, changed_state)
        return

    from IPython.display import display

    # Build 3×3 matrix of best Q values
    best_q_grid = np.zeros((3, 3))
    best_a_grid = np.empty((3, 3), dtype=object)

    for y in range(3):
        for x in range(3):
            s = xy_to_state(x, y)
            if s in HOLES:
                best_q_grid[y, x] = np.nan
                best_a_grid[y, x] = 'H'
            elif s == GOAL:
                best_q_grid[y, x] = np.max(q_table[s])
                best_a_grid[y, x] = 'G'
            else:
                ba = np.argmax(q_table[s])
                best_q_grid[y, x] = q_table[s][ba]
                best_a_grid[y, x] = ACTION_SYMBOLS[ba]

    # Create DataFrame: rows = y, cols = x
    df_val  = pd.DataFrame(best_q_grid,
                           index=[f'y={y}' for y in range(3)],
                           columns=[f'x={x}' for x in range(3)])

    df_act  = pd.DataFrame(best_a_grid,
                           index=[f'y={y}' for y in range(3)],
                           columns=[f'x={x}' for x in range(3)])

    # Combined label: "↓ +0.8145"
    df_label = pd.DataFrame(
        [[f"{best_a_grid[y,x]}  {best_q_grid[y,x]:+.4f}"
          if best_a_grid[y,x] not in ('H',) else 'HOLE'
          for x in range(3)]
         for y in range(3)],
        index=df_val.index,
        columns=df_val.columns
    )

    title_str = f"Optimal Q-Matrix — episode {episode}" \
                if episode else "Optimal Q-Matrix"
    print(f"\n  {title_str}")

    def color_cell(val):
        """Green for positive, red for negative, blue for near-zero."""
        if pd.isna(val):
            return 'background-color: #FCEBEB; color: #791F1F'
        if val > 0.01:
            intensity = min(int(val * 180), 160)
            return (f'background-color: rgb({234-intensity},{243-int(intensity*.7)},{222-intensity});'
                    f' color: #27500A; font-weight: 500')
        if val < -0.01:
            intensity = min(int(-val * 180), 160)
            return (f'background-color: rgb({252},{235-intensity},{235-intensity});'
                    f' color: #791F1F; font-weight: 500')
        return 'background-color: #E6F1FB; color: #0C447C'

    styled = (df_val.style
              .applymap(color_cell)
              .format(lambda v: f'{v:+.4f}' if not pd.isna(v) else 'HOLE')
              .set_caption(title_str)
              .set_table_styles([
                  {'selector': 'caption',
                   'props': [('font-size','13px'),('font-weight','500'),
                              ('text-align','left'),('padding-bottom','6px')]},
                  {'selector': 'th',
                   'props': [('font-size','12px'),('color','#555'),
                              ('background','#f5f5f5'),('padding','6px 14px')]},
                  {'selector': 'td',
                   'props': [('font-size','13px'),('padding','10px 18px'),
                              ('text-align','center'),('min-width','90px')]},
              ]))

    display(styled)

    # Also print best-action matrix below
    print("\n  Best actions:")
    df_act_display = pd.DataFrame(
        [[f"{best_a_grid[y,x]}" for x in range(3)] for y in range(3)],
        index=df_val.index, columns=df_val.columns
    )
    display(df_act_display.style.set_table_styles([
        {'selector':'td','props':[('font-size','18px'),('text-align','center'),
                                   ('padding','8px 20px')]},
        {'selector':'th','props':[('font-size','12px'),('padding','4px 20px'),
                                   ('background','#f5f5f5')]},
    ]))


def print_update_detail(ep, step_n, state, action, reward,
                        next_state, old_q, new_q, alpha, gamma, td_error, q_table):
    x0, y0 = state_to_xy(state)
    x1, y1 = state_to_xy(next_state)
    print(f"\n  ── Step {step_n}  (episode {ep}) ────────────────────────")
    print(f"  State      : (x={x0}, y={y0}) {GRID[y0][x0]}"
          f"  →  action {ACTION_SYMBOLS[action]} {ACTION_NAMES[action]}")
    print(f"  Next state : (x={x1}, y={y1}) {GRID[y1][x1]}"
          f"   reward = {reward:+.1f}")
    print(f"  TD error   : {reward} + {gamma}×{np.max(q_table[next_state]):.4f}"
          f" − {old_q:.4f} = {td_error:.4f}")
    print(f"  Update     : Q({x0},{y0},{ACTION_SYMBOLS[action]})"
          f"  {old_q:.4f} → {new_q:.4f}   (α={alpha})")

In [16]:
EPISODES      = 200
ALPHA         = 0.8
GAMMA         = 0.95
EPSILON_START = 1.0
EPSILON_END   = 0.05
EPSILON_DECAY = 0.99

PRINT_EVERY      = 5    # Print Q-matrix snapshot every N episodes
VERBOSE_UPDATES  = False  # True → print every single Bellman step
USE_PANDAS_STYLE = True   # True → coloured Pandas table in Colab output

q_table   = np.zeros((N_STATES, N_ACTIONS))
epsilon   = EPSILON_START
rewards_log  = []
success_log  = []

print("=" * 52)
print("  3×3 FROZEN LAKE — Q-LEARNING")
print("=" * 52)
print(f"  Coordinates:  x = column,  y = row")
print(f"  Episodes={EPISODES}   α={ALPHA}   γ={GAMMA}")
print(f"  ε: {EPSILON_START} → {EPSILON_END}  (decay {EPSILON_DECAY})")

# Initial matrix (all zeros)
if USE_PANDAS_STYLE and HAS_PANDAS:
    print_optimal_q_matrix_pandas(q_table, episode=0)
else:
    print_optimal_q_matrix(q_table, episode=0)

for ep in range(1, EPISODES + 1):
    state  = START
    done   = False
    total_r = 0
    step_n  = 0
    last_changed = None

    while not done:
        step_n += 1

        # ε-greedy
        if random.random() < epsilon:
            action = random.randint(0, N_ACTIONS - 1)
        else:
            action = np.argmax(q_table[state])

        next_state, reward, done = step(state, action)

        # Bellman update
        old_q    = q_table[state, action]
        td_error = reward + GAMMA * np.max(q_table[next_state]) - old_q
        new_q    = old_q + ALPHA * td_error
        q_table[state, action] = new_q
        last_changed = state

        if VERBOSE_UPDATES:
            print_update_detail(ep, step_n, state, action, reward,
                                next_state, old_q, new_q,
                                ALPHA, GAMMA, td_error, q_table)

        total_r += reward
        state    = next_state

    rewards_log.append(total_r)
    success_log.append(1 if total_r > 0 else 0)
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)

    # Snapshot every PRINT_EVERY episodes
    if ep % PRINT_EVERY == 0:
        clear_output(wait=True)
        print(f"  Training... episode {ep}/{EPISODES}"
              f"  |  win rate (last {PRINT_EVERY}): "
              f"{100*sum(success_log[-PRINT_EVERY:])/PRINT_EVERY:.0f}%"
              f"  |  ε={epsilon:.3f}")

        if USE_PANDAS_STYLE and HAS_PANDAS:
            print_optimal_q_matrix_pandas(q_table, episode=ep,
                                          changed_state=last_changed)
        else:
            print_optimal_q_matrix(q_table, episode=ep,
                                   changed_state=last_changed)
        time.sleep(0.3)

  Training... episode 200/200  |  win rate (last 5): 100%  |  ε=0.134

  Optimal Q-Matrix — episode 200


,x=0,x=1,x=2
y=0,+0.8574,+0.9025,+0.9500
y=1,+0.9025,HOLE,+1.0000
y=2,+0.9500,+1.0000,+0.0000



  Best actions:


,x=0,x=1,x=2
y=0,→,→,↓
y=1,↓,H,↓
y=2,→,→,G


In [17]:
clear_output(wait=True)
print("=" * 52)
print("  TRAINING COMPLETE")
print("=" * 52)
print(f"  Episodes : {EPISODES}")
print(f"  Final ε  : {epsilon:.4f}")
print(f"  Win rate (last 100): {100*sum(success_log[-100:])/100:.0f}%")

if USE_PANDAS_STYLE and HAS_PANDAS:
    print_optimal_q_matrix_pandas(q_table, episode=EPISODES)
else:
    print_optimal_q_matrix(q_table, episode=EPISODES)

# ── Optimal path ──
print("\n" + "=" * 52)
print("  OPTIMAL PATH")
print("=" * 52)

state = START
path  = [state]
acts  = []
done  = False

while not done and len(path) < 20:
    action = np.argmax(q_table[state])
    ns, r, done = step(state, action)
    acts.append(action)
    path.append(ns)
    state = ns

print("\n  Step-by-step:")
for i, (s, a) in enumerate(zip(path[:-1], acts)):
    x, y = state_to_xy(s)
    print(f"  Step {i+1}: (x={x}, y={y}) {GRID[y][x]}"
          f"  →  {ACTION_SYMBOLS[a]} {ACTION_NAMES[a]}")

fx, fy = state_to_xy(path[-1])
outcome = "✓ GOAL reached!" if path[-1] == GOAL else "✗ Did not reach goal"
print(f"\n  Final: (x={fx}, y={fy}) {GRID[fy][fx]}  —  {outcome}")
print(f"\n  Path (x,y): "
      + "  →  ".join(f"({state_to_xy(s)[0]},{state_to_xy(s)[1]})" for s in path))
print(f"  Actions  : {' → '.join(ACTION_SYMBOLS[a] for a in acts)}")

# ── Reward summary ──
print("\n" + "=" * 52)
print("  REWARD SUMMARY (50-episode windows)")
print("=" * 52)
window = 50
for start in range(0, EPISODES, window):
    chunk = rewards_log[start:start+window]
    avg   = sum(chunk) / len(chunk)
    w     = sum(1 for x in chunk if x > 0)
    bar   = '█' * w + '░' * (len(chunk)-w)
    print(f"  Ep {start+1:>3}–{start+len(chunk):<3}  avg {avg:+.2f}"
          f"  wins {w:>2}/{len(chunk)}  {bar}")

print("\n  Done!\n")

  TRAINING COMPLETE
  Episodes : 200
  Final ε  : 0.1340
  Win rate (last 100): 97%

  Optimal Q-Matrix — episode 200


/tmp/ipykernel_1566/902332543.py:122: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(color_cell)


,x=0,x=1,x=2
y=0,+0.8574,+0.9025,+0.9500
y=1,+0.9025,HOLE,+1.0000
y=2,+0.9500,+1.0000,+0.0000



  Best actions:


,x=0,x=1,x=2
y=0,→,→,↓
y=1,↓,H,↓
y=2,→,→,G



  OPTIMAL PATH

  Step-by-step:
  Step 1: (x=0, y=0) S  →  → Right
  Step 2: (x=1, y=0) F  →  → Right
  Step 3: (x=2, y=0) F  →  ↓ Down
  Step 4: (x=2, y=1) F  →  ↓ Down

  Final: (x=2, y=2) G  —  ✓ GOAL reached!

  Path (x,y): (0,0)  →  (1,0)  →  (2,0)  →  (2,1)  →  (2,2)
  Actions  : → → → → ↓ → ↓

  REWARD SUMMARY (50-episode windows)
  Ep   1–50   avg -0.36  wins 16/50  ████████████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  Ep  51–100  avg +0.24  wins 31/50  ███████████████████████████████░░░░░░░░░░░░░░░░░░░
  Ep 101–150  avg +0.92  wins 48/50  ████████████████████████████████████████████████░░
  Ep 151–200  avg +0.96  wins 49/50  █████████████████████████████████████████████████░

  Done!

